In [1]:
# directories
pdfs_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\pdfs'
ocr_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\ocr_text'
extracted_text_dir = 'C:\\Users\\samwt\\Documents\\ExtendedEssay\\data\\raw\\rowells_directories\\extracted_csv'

In [ ]:
# OCR parser Script 1877 - 1890

import fitz  # PyMuPDF
import os


def split_merged_line(line, midpoint):
    """
    Split a merged line into left and right column content by examining spans.
    """
    
    left_spans = []
    right_spans = []
    
    for span in line['spans']:
        span_bbox = span["bbox"]
        span_x0 = span_bbox[0]
        span_x1 = span_bbox[2]
        span_center = (span_x0 + span_x1) / 2
        span_text = span["text"]
        
        if not span_text.strip():
            continue
        
        # Classify span by its center position
        if span_center < midpoint:
            left_spans.append(span_text)
        else:
            right_spans.append(span_text)
    
    left_text = "".join(left_spans).strip()
    right_text = "".join(right_spans).strip()
    
    return left_text, right_text


def extract_columns_v11(
    pages_to_scan,
    input_dir=pdfs_dir,
    output_dir=ocr_text_dir,
    file_to_start_at=None
):
    """
    Two-column extraction with dynamic boundary detection per page.
    """
    
    os.makedirs(output_dir, exist_ok=True)
    years = sorted(pages_to_scan.keys())
    
    print(f"Will process years: {years}")
    
    if file_to_start_at is not None:
        years = [y for y in years if y >= file_to_start_at]
        print(f"Starting from year {file_to_start_at}")
    
    for year in years:
        start_page, end_page = pages_to_scan[year]
        
        pdf_path = os.path.join(input_dir, f"Rowell {year}.pdf")
        if not os.path.exists(pdf_path):
            print(f"WARNING: {pdf_path} not found, skipping...")
            continue
        
        output_path = os.path.join(output_dir, f"Rowell {year} - v13.txt")
        
        if os.path.exists(output_path):
            print(f"SKIPPING {year}: Output already exists")
            continue
        
        print(f"\n{'='*60}")
        print(f"Processing: Rowell {year}.pdf (pages {start_page}-{end_page})")
        print(f"{'='*60}")
        
        pdf = fitz.open(pdf_path)
        all_text = []
        
        for page_num in range(start_page - 1, min(end_page, len(pdf))):
            page = pdf[page_num]
            page_width = page.rect.width
            
            page_text = process_page_v11(page, page_width, page_num + 1)
            all_text.append(page_text)
            
            pages_done = page_num - (start_page - 1) + 1
            total_pages = end_page - start_page + 1
            if pages_done % 50 == 0 or pages_done == total_pages:
                print(f"  {pages_done}/{total_pages} pages ({100*pages_done/total_pages:.1f}%)")
        
        pdf.close()
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(all_text))
        
        print(f"  SAVED: {output_path}")
    
    print(f"\n{'='*60}")
    print("Done!")


def process_page_v11(page, page_width, page_num):
    """
    Process page with dynamic column boundary detection.
    """
    
    text_dict = page.get_text("dict")
    midpoint = page_width / 2
    
    # Collect all lines
    all_lines = []
    for block in text_dict["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            line_bbox = line["bbox"]
            line_text = "".join(span["text"] for span in line["spans"]).strip()
            if not line_text:
                continue
            all_lines.append({
                'text': line_text,
                'x0': line_bbox[0],
                'x1': line_bbox[2],
                'y': line_bbox[1],
                'spans': line["spans"],  # Keep spans for potential splitting
            })
    
    if not all_lines:
        return f"--- Page {page_num} ---\n"
    
    # Find the column boundary by analyzing line positions
    clear_left_x1 = []
    for line in all_lines:
        if line['x1'] < midpoint * 0.95 and line['x0'] < midpoint * 0.5:
            clear_left_x1.append(line['x1'])
    
    clear_right_x0 = []
    for line in all_lines:
        if line['x0'] > midpoint * 1.05:
            clear_right_x0.append(line['x0'])
    
    if clear_left_x1 and clear_right_x0:
        left_edge = max(clear_left_x1)
        right_edge = min(clear_right_x0)
        if right_edge > left_edge:
            boundary = (left_edge + right_edge) / 2
        else:
            boundary = right_edge - 5
    else:
        boundary = midpoint
    
    # Classify lines using the computed boundary
    left_lines = []
    right_lines = []


    # Threshold for "line extends into right column territory"
    right_extent_threshold = midpoint * 1.4
    
    # Threshold for "short line" - if a line is shorter than the gap between midpoint and right_extent_threshold
    short_line_threshold = right_extent_threshold - midpoint + 10  # e.g., 252.8 - 180.6 = 72.2
    
    for line in all_lines:
        line_width = line['x1'] - line['x0']
        
        # Check if this line spans both columns (merged line)
        # Criteria: starts in left column territory AND extends well past midpoint
        starts_left = line['x0'] < midpoint * 0.8
        extends_right = line['x1'] > midpoint * 1.3
        is_wide = line_width > page_width * 0.5
        
        if starts_left and extends_right and is_wide:
            # This is likely a merged line - split it by examining spans
            left_text, right_text = split_merged_line(line, midpoint)
            
            if left_text:
                left_lines.append({
                    'text': left_text,
                    'y': line['y'],
                })
            if right_text:
                right_lines.append({
                    'text': right_text,
                    'y': line['y'],
                })
        # If line STARTS past boundary, it's right column
        elif line['x0'] >= boundary:
            right_lines.append(line)
        # If line STARTS left of boundary BUT EXTENDS well into right territory,
        # it's likely right column content with a slightly left-shifted start due to slant
        elif line['x1'] > right_extent_threshold and line['x0'] > midpoint * 0.9:
            right_lines.append(line)
        # Short lines that start before boundary but extend past it - likely right column
        # Must have at least 40% of line width past the boundary
        elif (line_width < short_line_threshold 
              and line['x0'] < boundary 
              and line['x1'] > boundary
              and (line['x1'] - boundary) / line_width >= 0.35):
            right_lines.append(line)
        else:
            left_lines.append(line)
    
    # Sort each column by y-position
    left_lines.sort(key=lambda l: l['y'])
    right_lines.sort(key=lambda l: l['y'])

    def fix_same_row_order(lines):
        if len(lines) < 2:
            return lines
        
        i = 0
        while i < len(lines) - 1:
            curr = lines[i]
            next_line = lines[i + 1]
            
            # Skip if either line is missing x0/x1 (e.g., from split lines)
            if 'x0' not in curr or 'x0' not in next_line or 'x1' not in next_line:
                i += 1
                continue
            
            # If lines are on same visual row (within 2 units of Y)
            # and next line starts to the LEFT of current line
            # and they are horizontally adjacent (curr x0 is within 20 units of next_line x1)
            if (abs(curr['y'] - next_line['y']) <= 2 
                and next_line['x0'] < curr['x0']
                and abs(curr['x0'] - next_line['x1']) <= 20):
                # Swap them
                lines[i], lines[i + 1] = lines[i + 1], lines[i]
                # Check if we need to swap backwards too
                if i > 0:
                    i -= 1
                    continue
            i += 1
        
        return lines
    
    left_lines = fix_same_row_order(left_lines)
    right_lines = fix_same_row_order(right_lines)
    
    # Build output
    result = [f"--- Page {page_num} ---"]
    for line in left_lines:
        result.append(line['text'])
    for line in right_lines:
        result.append(line['text'])
    
    return '\n'.join(result)


def debug_page(pdf_path, page_num, search_text=None):
    """Debug: show boundary detection and line classifications."""
    
    pdf = fitz.open(pdf_path)
    page = pdf[page_num - 1]
    page_width = page.rect.width
    midpoint = page_width / 2
    
    text_dict = page.get_text("dict")
    
    all_lines = []
    for block in text_dict["blocks"]:
        if block["type"] != 0:
            continue
        for line in block["lines"]:
            line_bbox = line["bbox"]
            line_text = "".join(span["text"] for span in line["spans"]).strip()
            if not line_text:
                continue
            all_lines.append({
                'text': line_text,
                'x0': line_bbox[0],
                'x1': line_bbox[2],
                'y': line_bbox[1],
                'spans': line["spans"],
            })
    
    # Find boundary
    clear_left_x1 = []
    for line in all_lines:
        if line['x1'] < midpoint * 0.95 and line['x0'] < midpoint * 0.5:
            clear_left_x1.append(line['x1'])
    
    clear_right_x0 = []
    for line in all_lines:
        if line['x0'] > midpoint * 1.05:
            clear_right_x0.append(line['x0'])
    
    if clear_left_x1 and clear_right_x0:
        left_edge = max(clear_left_x1)
        right_edge = min(clear_right_x0)
        if right_edge > left_edge:
            boundary = (left_edge + right_edge) / 2
        else:
            boundary = right_edge - 5
    else:
        boundary = midpoint
    
    # Thresholds
    right_extent_threshold = midpoint * 1.4
    short_line_threshold = right_extent_threshold - midpoint + 10
    
    print(f"Page {page_num}")
    print(f"  Page width: {page_width:.1f}")
    print(f"  Midpoint: {midpoint:.1f}")
    print(f"  Left col max x1: {max(clear_left_x1):.1f}" if clear_left_x1 else "  No clear left lines")
    print(f"  Right col min x0: {min(clear_right_x0):.1f}" if clear_right_x0 else "  No clear right lines")
    print(f"  Computed boundary: {boundary:.1f}")
    print(f"  Right extent threshold: {right_extent_threshold:.1f}")
    print(f"  Short line threshold: {short_line_threshold:.1f}")
    print("="*100)
    
    # Show lines sorted by y
    all_lines.sort(key=lambda l: l['y'])
    
    print(f"{'Y':>6} {'X0':>6} {'X1':>6} {'WIDTH':>6} {'%PAST':>6} {'COL':>6} | Text (first 60 chars)")
    print("-"*100)
    
    for line in all_lines:
        line_width = line['x1'] - line['x0']
        
        # Calculate percent past boundary (if applicable)
        if line['x1'] > boundary and line_width > 0:
            pct_past = (line['x1'] - boundary) / line_width * 100
        else:
            pct_past = 0
        
        # Check for merged line
        starts_left = line['x0'] < midpoint * 0.8
        extends_right = line['x1'] > midpoint * 1.3
        is_wide = line_width > page_width * 0.5
        
        if starts_left and extends_right and is_wide:
            col = "SPLIT"
            left_text, right_text = split_merged_line(line, midpoint)
            text_preview = f"L:[{left_text[:25]}] R:[{right_text[:25]}]"
        elif line['x0'] >= boundary:
            col = "RIGHT"
            text_preview = line['text'][:60]
        elif line['x1'] > right_extent_threshold and line['x0'] > midpoint * 0.9:
            col = "RIGHT"
            text_preview = line['text'][:60]
        elif (line_width < short_line_threshold 
              and line['x0'] < boundary 
              and line['x1'] > boundary
              and (line['x1'] - boundary) / line_width >= 0.35):
            col = "RIGHT*"
            text_preview = line['text'][:60]
        else:
            col = "LEFT"
            text_preview = line['text'][:60]
        
        # Highlight search text if provided
        marker = ">>>" if search_text and search_text.upper() in line['text'].upper() else "   "
        print(f"{marker} {line['y']:6.1f} {line['x0']:6.1f} {line['x1']:6.1f} {line_width:6.1f} {pct_past:5.1f}% {col:>6} | {text_preview}")
    
    print(f"\nTotal lines: {len(all_lines)}")
    
    pdf.close()


def test_page(pdf_path, page_num):
    """Test on a single page."""
    
    pdf = fitz.open(pdf_path)
    page = pdf[page_num - 1]
    
    result = process_page_v11(page, page.rect.width, page_num)
    
    print(result)
    
    pdf.close()


def find_text_in_pdf(pdf_path, search_text, start_page=1, end_page=None):
    """Find which page(s) contain specific text."""
    
    pdf = fitz.open(pdf_path)
    if end_page is None:
        end_page = len(pdf)
    
    for page_num in range(start_page - 1, min(end_page, len(pdf))):
        page = pdf[page_num]
        text = page.get_text("text")
        if search_text.upper() in text.upper():
            print(f"Found '{search_text}' on page {page_num + 1}")
    
    pdf.close()


# USAGE

pages_to_scan = {
    1877: (17, 341),
    1878: (13, 341),
    1879: (26, 374),
    1880: (23, 404),
    1882: (15, 476),
    1883: (15, 452),
    1884: (15, 524),
    1885: (35, 590),
    1890: (63, 731),
}

# Debug specific pages:
# debug_page(pdfs_dir + r"/Rowell 1890.pdf", 375, "FOLSOM")

# Test specific pages:
# test_page(pdfs_dir + r"/Rowell 1890.pdf", 375)

# Run full extraction:
extract_columns_v11(pages_to_scan)

Will process years: [1877, 1878, 1879, 1880, 1882, 1883, 1884, 1885, 1890]

Processing: Rowell 1877.pdf (pages 17-341)
  50/325 pages (15.4%)
  100/325 pages (30.8%)
  150/325 pages (46.2%)
  200/325 pages (61.5%)
  250/325 pages (76.9%)
  300/325 pages (92.3%)
  325/325 pages (100.0%)
  SAVED: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1877 - v13.txt

Processing: Rowell 1878.pdf (pages 13-341)
  50/329 pages (15.2%)


In [2]:
# page scanner 1869 - 1876

# %% Imports and setup
import fitz  # pymupdf
import json
import re
from pathlib import Path
from datetime import datetime

# Page ranges for each year: {year: (start_page, end_page)}
# Note: These are 1-indexed page numbers (as they appear in the PDF)
pages_to_scan = {
    1869: (15, 187),
    1871: (21, 251),
    1872: (15, 290),
    1873: (35, 238),
    1876: (25, 260),
}

# %% Configuration - EDIT THESE
INPUT_FOLDER = Path(pdfs_dir)  # Path to folder containing the Rowell PDFs  
OUTPUT_FOLDER = Path(ocr_text_dir)  # Path for output text files (one per PDF)

PROGRESS_FILE = Path("extract_progress.json")

# %% Functions
def load_progress():
    if PROGRESS_FILE.exists():
        return set(json.loads(PROGRESS_FILE.read_text()))
    return set()

def save_progress(completed: set):
    PROGRESS_FILE.write_text(json.dumps(list(completed)))

def clear_progress():
    """Call this if you want to start fresh."""
    if PROGRESS_FILE.exists():
        PROGRESS_FILE.unlink()
        print("Progress cleared. Will reprocess all files.")
    else:
        print("No progress file found.")

def extract_year_from_filename(filename: str) -> int | None:
    """Extract the year (18XX) from a filename like 'Rowell 1871.pdf'."""
    match = re.search(r'18\d{2}', filename)
    if match:
        return int(match.group())
    return None

def extract_text_from_pdf(pdf_path: Path, output_path: Path, start_page: int, end_page: int):
    """Extract embedded text from PDF.
    
    Args:
        pdf_path: Path to the PDF file
        output_path: Path for the output text file
        start_page: First page to process (1-indexed)
        end_page: Last page to process (1-indexed, inclusive)
    """
    print(f"\nProcessing: {pdf_path.name}")
    
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    
    # Convert to 0-indexed and clamp to valid range
    start_idx = max(0, start_page - 1)
    end_idx = min(total_pages, end_page)
    
    print(f"  Total pages in PDF: {total_pages}")
    print(f"  Extracting pages {start_idx + 1} to {end_idx} ({end_idx - start_idx} pages)")
    
    all_text = []
    
    for page_num in range(start_idx, end_idx):
        text = doc[page_num].get_text()
        all_text.append(f"--- Page {page_num + 1} ---\n{text}")
    
    doc.close()
    
    # Save output
    output_path.write_text("\n\n".join(all_text), encoding="utf-8")
    print(f"  Saved: {output_path}")

# %% Run extraction - THIS IS THE MAIN CELL
INPUT_FOLDER.mkdir(exist_ok=True)
OUTPUT_FOLDER.mkdir(exist_ok=True)

# Find PDFs and check progress
pdfs = sorted(INPUT_FOLDER.glob("*.pdf"))
completed = load_progress()

# Filter to only PDFs we have page ranges for
valid_pdfs = []
skipped_no_range = []

for p in pdfs:
    year = extract_year_from_filename(p.name)
    if year and year in pages_to_scan:
        if p.name not in completed:
            valid_pdfs.append((p, year))
    else:
        skipped_no_range.append(p.name)

skipped_completed = [p.name for p in pdfs if p.name in completed]

print(f"Found {len(pdfs)} PDF(s) total")
print(f"  Already completed: {len(skipped_completed)}")
print(f"  Pending (with page ranges): {len(valid_pdfs)}")
if skipped_no_range:
    print(f"  Skipped (no page range defined): {len(skipped_no_range)}")
    for name in skipped_no_range:
        print(f"    - {name}")

if not valid_pdfs:
    print("\nNo files to process. Run clear_progress() if you want to start over.")
else:
    print(f"\nWill process:")
    for p, year in valid_pdfs:
        start, end = pages_to_scan[year]
        print(f"  - {p.name} (pages {start}-{end})")
    
    # Process all pending PDFs
    start_time = datetime.now()
    
    for i, (pdf_path, year) in enumerate(valid_pdfs, 1):
        print(f"\n{'='*60}")
        print(f"[{i}/{len(valid_pdfs)}]", end=" ")
        
        output_path = OUTPUT_FOLDER / f"{pdf_path.stem}.txt"
        start_page, end_page = pages_to_scan[year]
        
        extract_text_from_pdf(pdf_path, output_path, start_page=start_page, end_page=end_page)
        
        # Mark as completed and save progress immediately
        completed.add(pdf_path.name)
        save_progress(completed)
        print(f"  ✓ Progress saved")
    
    # Summary
    elapsed = datetime.now() - start_time
    print(f"\n{'='*60}")
    print(f"Complete! Processed {len(valid_pdfs)} file(s) in {elapsed}")
    print(f"Output saved to '{OUTPUT_FOLDER}' folder")

# %% Utility: Clear progress and start over (run manually if needed)
# clear_progress()

Found 14 PDF(s) total
  Already completed: 0
  Pending (with page ranges): 5
  Skipped (no page range defined): 9
    - Rowell 1877.pdf
    - Rowell 1878.pdf
    - Rowell 1879.pdf
    - Rowell 1880.pdf
    - Rowell 1882.pdf
    - Rowell 1883.pdf
    - Rowell 1884.pdf
    - Rowell 1885.pdf
    - Rowell 1890.pdf

Will process:
  - Rowell 1869.pdf (pages 15-187)
  - Rowell 1871.pdf (pages 21-251)
  - Rowell 1872.pdf (pages 15-290)
  - Rowell 1873.pdf (pages 35-238)
  - Rowell 1876.pdf (pages 25-260)

[1/5] 
Processing: Rowell 1869.pdf
  Total pages in PDF: 387
  Extracting pages 15 to 187 (173 pages)
  Saved: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1869.txt
  ✓ Progress saved

[2/5] 
Processing: Rowell 1871.pdf
  Total pages in PDF: 593
  Extracting pages 21 to 251 (231 pages)
  Saved: C:\Users\samwt\Documents\ExtendedEssay\data\raw\rowells_directories\ocr_text\Rowell 1871.txt
  ✓ Progress saved

[3/5] 
Processing: Rowell 1872.pdf
  Total pages 